[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/050.%20The%20New%20Product%20Forecasting%20Problem/P50-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 50. The New Product Forecasting Problem: Look-Alike Modeling

## Tier 7: The Human–AI Partnership

### Purpose and scope

This notebook demonstrates a human–AI workflow for forecasting demand for a new product. An AI component supplies similarity scores for existing products and a baseline forecast. Domain experts review the analogs and provide contextual market information, which the workflow applies to the baseline forecast.

The example is a transparent simulation, not a trained forecasting model or a measured evaluation of forecast accuracy. Expert adjustments, contextual factors, confidence, explainability, and consensus are represented by illustrative rules or inputs. No actual demand observation is supplied, so forecast error and accuracy improvement cannot be calculated from this notebook.

### Modeling assumptions

- The AI similarity scores and baseline forecast are supplied as inputs; this notebook does not train the AI model.
- Expert judgments are simulated using role-based rules and a random noise term. Consequently, similarity adjustments can vary between runs.
- Contextual impacts are applied sequentially as multiplicative adjustments. Their magnitudes and confidence values are illustrative inputs, not statistically estimated effects.
- Confidence, explainability, and consensus scores are heuristic indicators. They are not calibrated probabilities or empirical performance measurements.
- The workflow records session results but does not update expert weights from observed forecast errors. Continuous learning is therefore a design consideration, not an implemented capability.
- Operational constraints such as product availability, price changes, and inventory limits are outside the scope of this example.

### Workflow

1. **Expert team assembly:** Represent experts by their roles, experience, expertise, and decision weights.
2. **Enhanced similarity assessment:** Combine AI similarity scores with simulated expert judgments.
3. **Analog selection and validation:** Rank products by enhanced similarity and present the leading candidates for validation.
4. **Contextual forecast generation:** Gather market intelligence represented by signed impact factors.
5. **Collaborative refinement:** Apply each factor to the baseline forecast, then apply a consensus adjustment.
6. **Performance review:** Report forecast changes and heuristic collaboration metrics. Accuracy requires an actual demand value and is not inferred from the forecast change.

### Similarity aggregation

For an AI-suggested analog product \(j\), let \(s_j\) be its AI similarity score. Expert \(e\) produces an adjusted score \(a_{je}\), and has a nonnegative decision weight \(w_e\). The combined similarity is the weighted mean

$$
\bar{a}_j = \frac{\sum_{e \in E} w_e a_{je}}{\sum_{e \in E} w_e},
$$

where \(E\) is the expert team. The simulated expert score is formed by adding a role-based adjustment and random noise to the AI score, then clipping the result to the interval \([0,1]\). An expert-suggested product not present among the AI inputs can also be added with its stated illustrative score. The code implements these steps in `_simulate_expert_adjustment` and `_enhance_similarity_with_experts`.

### Contextual forecast refinement

For market factor \(k\), let \(m_k\) be its nonnegative magnitude, \(c_k \in [0,1]\) its confidence, and \(q_k=+1\) for a positive impact or \(q_k=-1\) for a negative impact. Starting with baseline forecast \(F_0\), the sequential refinement is

$$
F_k = F_{k-1}(1 + q_k m_k c_k), \qquad k=1,\ldots,K.
$$

The final forecast applies the illustrative consensus multiplier \(1.05\):

$$
F_{\mathrm{final}} = 1.05 F_K.
$$

Here \(K\) is the number of contextual factors. Because the factors are applied sequentially, their combined effect is multiplicative rather than the sum of their individual percentage impacts. `_collaborative_refinement` implements this calculation. This rule is a demonstration of the stated workflow, not a claim that these adjustments are empirically optimal.

### Illustrative TechNova X1 inputs

The example uses a baseline forecast of 148,500 units and the following AI similarity scores: iPhone_Pro (0.87), Galaxy_Ultra (0.79), and Pixel_Advanced (0.71). The expert team and market factors are specified in the code. These inputs illustrate the collaboration process; they do not establish an observed forecast outcome or forecast accuracy.

### Interpretation

The final forecast is the output of the specified expert-adjustment and contextual-factor rules. A higher or lower forecast is a change relative to the baseline, not evidence of improved accuracy. To evaluate accuracy, compare forecasts with actual demand using an explicitly chosen error measure. Results involving random expert noise may differ across runs.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A consistent plotting style makes comparisons easier to read.
plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class DomainExpert:
    """Describe an expert participating in a forecasting session."""

    name: str
    role: str
    experience_years: int
    expertise_areas: List[str]
    confidence_weight: float
    availability_hours: int


@dataclass
class AIModel:
    """Store descriptive information about the AI forecasting component."""

    model_name: str
    accuracy_score: float
    confidence_interval: Tuple[float, float]
    processing_time_minutes: int
    feature_importance: Dict[str, float]


@dataclass
class MarketIntelligence:
    """Represent one contextual market factor supplied for refinement."""

    factor_name: str
    impact_direction: str
    magnitude: float
    confidence: float
    time_horizon: str


@dataclass
class SymbioticResult:
    """Collect the outputs and descriptive metrics from one session."""

    ai_forecast: float
    human_adjusted_forecast: float
    final_symbiotic_forecast: float
    confidence_score: float
    explainability_score: float
    expert_consensus: float
    selected_analogs: List[str]
    adjustment_factors: List[MarketIntelligence]
    computation_time: float
    expert_time_investment: float

## Representing the partnership

The partnership object coordinates the workflow and retains session results. Each stage remains a named method so that the sequence—from similarity assessment through forecast refinement and reporting—can be inspected and followed independently. The `learning_weights` dictionary records a placeholder for possible future feedback-based adaptation; this example does not update those weights because no observed demand or forecast-error feedback is provided.

In [ ]:
class HumanAISymbioticPartnership:
    """Coordinate human–AI collaboration for look-alike forecasting."""

    def __init__(self, ai_model: AIModel, experts: List[DomainExpert]):
        self.ai_model = ai_model
        self.experts = experts
        self.partnership_history = []
        self.learning_weights = {expert.name: 1.0 for expert in experts}

    def collaborative_forecasting_session(
        self,
        product_name: str,
        ai_similarity_scores: Dict[str, float],
        base_forecast: float,
    ) -> SymbioticResult:
        """Run the stages of a collaborative forecasting session in order."""
        start_time = time.perf_counter()

        print(f"\nCOLLABORATIVE FORECASTING SESSION: {product_name}")
        print("=" * 60)

        print("\nPhase 1: Enhanced Similarity Assessment")
        enhanced_similarities = self._enhance_similarity_with_experts(
            ai_similarity_scores
        )

        print("\nPhase 2: Analog Selection Validation")
        validated_analogs = self._validate_analog_selection(
            enhanced_similarities
        )

        print("\nPhase 3: Contextual Forecast Generation")
        contextual_factors = self._gather_market_intelligence()

        print("\nPhase 4: Collaborative Refinement")
        refined_forecast = self._collaborative_refinement(
            base_forecast, contextual_factors
        )

        confidence_score = self._calculate_confidence_score(contextual_factors)
        explainability_score = self._calculate_explainability_score()
        expert_consensus = self._calculate_expert_consensus()

        result = SymbioticResult(
            ai_forecast=base_forecast,
            human_adjusted_forecast=refined_forecast["base_adjusted"],
            final_symbiotic_forecast=refined_forecast["final"],
            confidence_score=confidence_score,
            explainability_score=explainability_score,
            expert_consensus=expert_consensus,
            selected_analogs=validated_analogs,
            adjustment_factors=contextual_factors,
            computation_time=time.perf_counter() - start_time,
            expert_time_investment=len(self.experts) * 0.75,
        )

        # Retaining session records supports later review; it does not itself
        # update weights or learn from forecast errors.
        self.partnership_history.append(result)
        return result

In [ ]:
def _enhance_similarity_with_experts(
    self, ai_similarities: Dict[str, float]
) -> Dict[str, float]:
    """Aggregate simulated expert judgments into enhanced analog scores."""
    expert_scores: Dict[str, List[float]] = {
        product: [] for product in ai_similarities
    }

    print("  AI similarity scores:")
    for product, score in ai_similarities.items():
        print(f"    {product}: {score:.2f}")

    print("\n  Simulated expert adjustments:")
    for expert in self.experts:
        print(f"\n    {expert.name} ({expert.role}):")
        for product, ai_score in ai_similarities.items():
            adjusted_score = self._simulate_expert_adjustment(
                expert, product, ai_score
            )
            expert_scores[product].append(adjusted_score)

            if abs(adjusted_score - ai_score) > 0.05:
                direction = "up" if adjusted_score > ai_score else "down"
                reasoning = self._generate_expert_reasoning(
                    expert, product, adjusted_score, ai_score
                )
                print(
                    f"      {product}: {ai_score:.2f} to "
                    f"{adjusted_score:.2f} ({direction})"
                )
                print(f"        Rationale: {reasoning}")

    final_similarities = {}
    for product, ai_score in ai_similarities.items():
        scores = expert_scores[product]
        if scores and self.experts:
            weights = [expert.confidence_weight for expert in self.experts]
            weight_sum = sum(weights)
            if weight_sum > 0:
                final_similarities[product] = sum(
                    score * weight
                    for score, weight in zip(scores, weights)
                ) / weight_sum
            else:
                final_similarities[product] = ai_score
        else:
            final_similarities[product] = ai_score

    # The regional sales team explicitly lists OnePlus in its expertise;
    # this rule represents an expert-suggested analog absent from AI inputs.
    if "OnePlus_Elite" not in ai_similarities and any(
        "OnePlus" in expert.expertise_areas for expert in self.experts
    ):
        final_similarities["OnePlus_Elite"] = 0.76
        print("\n    Expert-suggested analog: OnePlus_Elite (0.76)")

    return final_similarities


HumanAISymbioticPartnership._enhance_similarity_with_experts = (
    _enhance_similarity_with_experts
)

In [ ]:
def _simulate_expert_adjustment(
    self, expert: DomainExpert, product: str, ai_score: float
) -> float:
    """Apply an illustrative role-based adjustment and experience-scaled noise."""
    base_adjustment = 0.0

    if "Product Manager" in expert.role:
        if "iPhone" in product:
            base_adjustment = -0.05
        elif "Galaxy" in product:
            base_adjustment = 0.05
    elif "Market Research" in expert.role:
        if "Pixel" in product:
            base_adjustment = -0.03
        elif "OnePlus" in product:
            base_adjustment = 0.08

    experience_factor = expert.experience_years / 20.0
    noise = np.random.normal(0, 0.02 * (1 - experience_factor))
    return float(np.clip(ai_score + base_adjustment + noise, 0.0, 1.0))


def _generate_expert_reasoning(
    self,
    expert: DomainExpert,
    product: str,
    adjustment: float,
    ai_score: float,
) -> str:
    """Return an illustrative rationale for a reported score adjustment."""
    reasons = {
        "iPhone": "Different pricing strategy may reduce similarity.",
        "Galaxy": "Premium positioning and the Android ecosystem may be relevant.",
        "Pixel": "Market share and customer base may differ.",
        "OnePlus": "The product may share a target demographic overlooked by the algorithm.",
        "Xiaomi": "Supply-chain constraints may affect market positioning.",
        "Oppo": "Supply-chain strength and market presence may be relevant.",
    }
    for key, reason in reasons.items():
        if key in product:
            return reason
    return "Illustrative adjustment based on the expert's stated role and knowledge."


HumanAISymbioticPartnership._simulate_expert_adjustment = (
    _simulate_expert_adjustment
)
HumanAISymbioticPartnership._generate_expert_reasoning = (
    _generate_expert_reasoning
)

In [ ]:
def _validate_analog_selection(
    self, enhanced_similarities: Dict[str, float]
) -> List[str]:
    """Rank enhanced scores and return the three leading analog candidates."""
    sorted_products = sorted(
        enhanced_similarities.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    print("  Enhanced similarity rankings:")
    for rank, (product, score) in enumerate(sorted_products[:5], start=1):
        print(f"    {rank}. {product}: {score:.2f}")

    top_analogs = [product for product, _ in sorted_products[:3]]
    print(f"\n  Leading candidates for expert validation: {top_analogs}")
    if top_analogs and "Galaxy" in top_analogs[0]:
        print(
            "  Rationale: premium positioning and the Android ecosystem "
            "are relevant considerations."
        )
    return top_analogs


def _gather_market_intelligence(self) -> List[MarketIntelligence]:
    """Return the illustrative contextual factors used in forecast refinement."""
    factors = [
        MarketIntelligence("Seasonal Factor", "positive", 0.05, 0.9, "short"),
        MarketIntelligence("Competitive Pressure", "positive", 0.15, 0.8, "medium"),
        MarketIntelligence("Economic Conditions", "negative", 0.03, 0.7, "short"),
        MarketIntelligence("Technology Adoption", "positive", 0.22, 0.9, "medium"),
    ]

    print("  Illustrative market intelligence factors:")
    for factor in factors:
        print(
            f"    {factor.factor_name}: {factor.impact_direction} impact, "
            f"magnitude {factor.magnitude:.0%}, "
            f"confidence {factor.confidence:.0%}"
        )
    return factors


HumanAISymbioticPartnership._validate_analog_selection = (
    _validate_analog_selection
)
HumanAISymbioticPartnership._gather_market_intelligence = (
    _gather_market_intelligence
)

In [ ]:
def _collaborative_refinement(
    self,
    base_forecast: float,
    factors: List[MarketIntelligence],
) -> Dict[str, float]:
    """Apply signed, confidence-weighted factors sequentially to the forecast."""
    print("  Forecast refinement:")
    print(f"    Baseline AI forecast: {base_forecast:,.0f} units")

    adjusted_forecast = base_forecast
    for factor in factors:
        impact = factor.magnitude * factor.confidence
        if factor.impact_direction == "positive":
            adjustment_factor = 1.0 + impact
        elif factor.impact_direction == "negative":
            adjustment_factor = 1.0 - impact
        else:
            raise ValueError(
                "impact_direction must be 'positive' or 'negative'."
            )

        adjusted_forecast *= adjustment_factor
        print(
            f"    {factor.factor_name}: {factor.impact_direction} "
            f"{impact:.1%}; updated forecast {adjusted_forecast:,.0f} units"
        )

    final_forecast = adjusted_forecast * 1.05
    print(
        f"    Illustrative 5% consensus multiplier: "
        f"{final_forecast:,.0f} units"
    )
    return {
        "base": base_forecast,
        "base_adjusted": adjusted_forecast,
        "final": final_forecast,
    }


def _calculate_confidence_score(
    self, factors: List[MarketIntelligence]
) -> float:
    """Calculate the example's heuristic confidence indicator."""
    base_confidence = 0.78
    expert_boost = 0.13
    average_factor_confidence = (
        float(np.mean([factor.confidence for factor in factors])
        )
        if factors
        else 0.0
    )
    factor_boost = average_factor_confidence * 0.10
    return min(base_confidence + expert_boost + factor_boost, 0.95)


def _calculate_explainability_score(self) -> float:
    """Return the illustrative explainability indicator used in this example."""
    return 0.94


def _calculate_expert_consensus(self) -> float:
    """Return the illustrative consensus assumption used in this example."""
    return 0.85


HumanAISymbioticPartnership._collaborative_refinement = _collaborative_refinement
HumanAISymbioticPartnership._calculate_confidence_score = (
    _calculate_confidence_score
)
HumanAISymbioticPartnership._calculate_explainability_score = (
    _calculate_explainability_score
)
HumanAISymbioticPartnership._calculate_expert_consensus = (
    _calculate_expert_consensus
)

In [ ]:
def print_session_summary(
    self, result: SymbioticResult
) -> SymbioticResult:
    """Print the forecast, descriptive metrics, analogs, and factors."""
    print("\n" + "=" * 70)
    print("HUMAN–AI PARTNERSHIP: SESSION SUMMARY")
    print("=" * 70)

    print("\nForecast results:")
    print(f"  AI baseline forecast: {result.ai_forecast:,.0f} units")
    print(
        "  After contextual factors: "
        f"{result.human_adjusted_forecast:,.0f} units"
    )
    print(
        "  After consensus multiplier: "
        f"{result.final_symbiotic_forecast:,.0f} units"
    )
    if result.ai_forecast != 0:
        forecast_change = (
            (result.final_symbiotic_forecast - result.ai_forecast)
            / result.ai_forecast
            * 100
        )
        print(f"  Forecast change relative to baseline: {forecast_change:+.1f}%")
    print("  Forecast accuracy: not calculated; actual demand is not supplied.")

    print("\nHeuristic collaboration indicators:")
    print(f"  Confidence indicator: {result.confidence_score:.1%}")
    print(f"  Explainability indicator: {result.explainability_score:.1%}")
    print(f"  Consensus assumption: {result.expert_consensus:.1%}")
    print(f"  Computation time: {result.computation_time:.3f} seconds")
    print(f"  Expert time assumption: {result.expert_time_investment:.2f} hours")

    print("\nSelected analog candidates:")
    for product in result.selected_analogs:
        print(f"  {product}")

    print("\nExpert team:")
    for expert in self.experts:
        print(
            f"  {expert.name} ({expert.role}) — "
            f"{expert.experience_years} years of experience"
        )

    print("\nContextual factors applied:")
    for factor in result.adjustment_factors:
        print(
            f"  {factor.factor_name}: {factor.impact_direction}, "
            f"magnitude {factor.magnitude:.0%}, "
            f"confidence {factor.confidence:.0%}"
        )
    return result


HumanAISymbioticPartnership.print_session_summary = print_session_summary

## Assemble the example team and AI component

The following objects supply the inputs used by the session. The `accuracy_score` and confidence interval describe the example AI component; they are not independently evaluated in this notebook. Expert decision weights are used to aggregate expert similarity judgments.

In [ ]:
def create_symbiotic_system() -> HumanAISymbioticPartnership:
    """Create the illustrative expert team and AI component."""
    experts = [
        DomainExpert(
            name="Sarah Chen",
            role="Lead Product Manager",
            experience_years=15,
            expertise_areas=[
                "smartphone pricing",
                "product positioning",
                "feature analysis",
            ],
            confidence_weight=0.3,
            availability_hours=40,
        ),
        DomainExpert(
            name="Michael Rodriguez",
            role="Market Research Director",
            experience_years=12,
            expertise_areas=[
                "consumer trends",
                "market dynamics",
                "competitor analysis",
            ],
            confidence_weight=0.25,
            availability_hours=35,
        ),
        DomainExpert(
            name="Emily Watson",
            role="Supply Chain Analytics Expert",
            experience_years=10,
            expertise_areas=[
                "demand forecasting",
                "inventory optimization",
                "supply constraints",
            ],
            confidence_weight=0.2,
            availability_hours=45,
        ),
        DomainExpert(
            name="Regional Managers",
            role="Sales Intelligence",
            experience_years=8,
            expertise_areas=[
                "regional markets",
                "customer preferences",
                "OnePlus",
            ],
            confidence_weight=0.25,
            availability_hours=20,
        ),
    ]

    ai_model = AIModel(
        model_name="Advanced Look-Alike Neural Network",
        accuracy_score=0.912,
        confidence_interval=(0.878, 0.946),
        processing_time_minutes=15,
        feature_importance={
            "price": 0.25,
            "features": 0.20,
            "brand": 0.18,
            "market_position": 0.15,
            "timing": 0.12,
            "similarity": 0.10,
        },
    )
    return HumanAISymbioticPartnership(ai_model, experts)


partnership = create_symbiotic_system()
print(f"Expert team size: {len(partnership.experts)}")
print(f"AI component: {partnership.ai_model.model_name}")
print(
    "Total stated expert experience: "
    f"{sum(expert.experience_years for expert in partnership.experts)} years"
)

## Run the TechNova X1 session

This cell supplies the AI similarity scores and baseline forecast, then runs the partnership stages in sequence. The returned forecast is produced by the illustrative rules above. The random noise in simulated expert judgments means analog scores can vary across executions.

In [ ]:
def run_collaborative_session() -> SymbioticResult:
    """Run and summarize the illustrative TechNova X1 forecasting session."""
    product_name = "TechNova X1 Smartphone"
    ai_similarity_scores = {
        "iPhone_Pro": 0.87,
        "Galaxy_Ultra": 0.79,
        "Pixel_Advanced": 0.71,
    }
    base_forecast = 148500

    result = partnership.collaborative_forecasting_session(
        product_name,
        ai_similarity_scores,
        base_forecast,
    )
    return partnership.print_session_summary(result)


session_result = run_collaborative_session()

## Visualize the session

The plots distinguish forecast stages, heuristic indicators, expert-team attributes, and contextual factors. Positive and negative factor colors are identified in a legend. These visualizations describe the model inputs and outputs; they do not establish predictive accuracy.

In [ ]:
def create_symbiotic_visualizations(result: SymbioticResult):
    """Plot forecast stages, heuristic metrics, experts, and market factors."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Human–AI Partnership: Collaborative Forecasting",
        fontsize=16,
        fontweight="bold",
    )

    ax1 = axes[0, 0]
    forecast_labels = ["AI baseline", "After factors", "Final"]
    forecast_values = [
        result.ai_forecast,
        result.human_adjusted_forecast,
        result.final_symbiotic_forecast,
    ]
    bars = ax1.bar(
        forecast_labels,
        forecast_values,
        color=["lightblue", "orange", "green"],
        alpha=0.75,
    )
    ax1.set_title("Forecast through the collaboration stages")
    ax1.set_ylabel("Forecast (units)")
    ax1.grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, forecast_values):
        ax1.annotate(
            f"{value:,.0f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
        )

    ax2 = axes[0, 1]
    metric_names = ["Confidence", "Explainability", "Consensus"]
    metric_values = [
        result.confidence_score * 100,
        result.explainability_score * 100,
        result.expert_consensus * 100,
    ]
    metric_bars = ax2.bar(
        metric_names,
        metric_values,
        color=["steelblue", "mediumpurple", "darkseagreen"],
        alpha=0.8,
    )
    ax2.set_title("Heuristic collaboration indicators")
    ax2.set_ylabel("Indicator (%)")
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3, axis="y")
    for bar, value in zip(metric_bars, metric_values):
        ax2.annotate(
            f"{value:.1f}%",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
        )

    ax3 = axes[1, 0]
    expert_names = [expert.name for expert in partnership.experts]
    experience_years = [
        expert.experience_years for expert in partnership.experts
    ]
    confidence_weights = [
        expert.confidence_weight * 100 for expert in partnership.experts
    ]
    bubble_sizes = [max(weight * 10, 20) for weight in confidence_weights]
    ax3.scatter(
        experience_years,
        confidence_weights,
        s=bubble_sizes,
        alpha=0.7,
        label="Experts",
    )
    for name, years, weight in zip(
        expert_names, experience_years, confidence_weights
    ):
        ax3.annotate(
            name,
            (years, weight),
            xytext=(5, 5),
            textcoords="offset points",
        )
    ax3.set_title("Expert experience and decision weight")
    ax3.set_xlabel("Years of experience")
    ax3.set_ylabel("Decision weight (%)")
    ax3.grid(True, alpha=0.3)
    ax3.legend()

    ax4 = axes[1, 1]
    factor_names = [factor.factor_name for factor in result.adjustment_factors]
    signed_impacts = [
        factor.magnitude * factor.confidence * 100
        * (1 if factor.impact_direction == "positive" else -1)
        for factor in result.adjustment_factors
    ]
    factor_colors = [
        "green" if factor.impact_direction == "positive" else "red"
        for factor in result.adjustment_factors
    ]
    factor_bars = ax4.bar(
        factor_names,
        signed_impacts,
        color=factor_colors,
        alpha=0.75,
    )
    ax4.axhline(0, color="black", linewidth=0.8)
    ax4.set_title("Signed factor impact (magnitude × confidence)")
    ax4.set_ylabel("Signed adjustment (%)")
    ax4.tick_params(axis="x", rotation=35)
    ax4.grid(True, alpha=0.3, axis="y")
    for bar, value in zip(factor_bars, signed_impacts):
        vertical_alignment = "bottom" if value >= 0 else "top"
        offset = 2 if value >= 0 else -2
        ax4.annotate(
            f"{value:+.1f}%",
            (bar.get_x() + bar.get_width() / 2, value),
            xytext=(0, offset),
            textcoords="offset points",
            ha="center",
            va=vertical_alignment,
        )

    plt.tight_layout()
    plt.show()
    return fig


fig = create_symbiotic_visualizations(session_result)

## Summary and practical interpretation

The session demonstrates how expert judgments can modify analog similarity scores and how contextual information can adjust a baseline forecast. The example makes each stage visible, but its numerical indicators should be interpreted according to how they are constructed: the forecast is rule-based, the expert adjustments include random noise, and confidence, explainability, and consensus are assumptions rather than measured outcomes.

A forecast change is not an accuracy improvement. To assess accuracy, a practitioner must compare forecasts with observed demand over a defined evaluation period, using an explicitly stated error metric. A feedback process could then use those observations to assess expert contributions and recalibrate the workflow; that learning step is outside the implementation here.

### When this partnership design may be useful

- High-stakes product launches where business context should be reviewed alongside model outputs.
- Unusual market conditions that may not be represented in historical data.
- Decisions for which stakeholders need a traceable account of assumptions and adjustments.
- Organizations that can make relevant domain expertise available.

### Practical limitations

- Expert participation requires time and coordination.
- Role-based simulated adjustments do not represent actual expert elicitation and may encode bias.
- Sequential multiplication makes the final forecast sensitive to the factor values and their assumptions.
- The workflow does not estimate uncertainty from historical forecast errors or demonstrate that collaboration improves accuracy.
- The stored partnership history is not used to update weights or adapt the model.